In [91]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]

    stemmer = PorterStemmer()
    tokens = [stemmer.stem(t) for t in tokens]

    return ' '.join(tokens)

label_map = {'positive': 1, 'neutral': 0, 'negative': -1}

labels = []
tweets = []

with open('resources/tweets_train.txt', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        if ',' not in line:
            continue

        first_comma = line.index(',')
        label_str = line[:first_comma].strip().lower()
        tweet_text = line[first_comma+1:].strip()

        labels.append(label_map[label_str])
        tweets.append(tweet_text)

preprocessed = [preprocess(tweet) for tweet in tweets]

vectorizer = CountVectorizer(max_features=500)
count_matrix = vectorizer.fit_transform(preprocessed)


print(count_matrix)

count_vectorized_df = pd.DataFrame.sparse.from_spmatrix(
    count_matrix,
    columns=vectorizer.get_feature_names_out()
)

print(count_vectorized_df.iloc[:3,400-1:403-1])

[nltk_data] Downloading package punkt to /home/oelhaouc/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/oelhaouc/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 37334 stored elements and shape (6588, 500)>
  Coords	Values
  (0, 209)	1
  (0, 203)	1
  (0, 212)	1
  (0, 172)	1
  (0, 374)	1
  (1, 414)	1
  (1, 389)	1
  (1, 471)	1
  (1, 111)	1
  (1, 375)	1
  (2, 212)	1
  (2, 140)	1
  (2, 306)	1
  (2, 78)	1
  (2, 466)	1
  (2, 143)	1
  (3, 111)	1
  (3, 78)	1
  (3, 377)	1
  (3, 230)	1
  (3, 423)	1
  (3, 252)	1
  (3, 275)	1
  (3, 128)	1
  (3, 150)	1
  :	:
  (6583, 443)	1
  (6583, 454)	1
  (6583, 210)	1
  (6584, 286)	2
  (6584, 308)	1
  (6584, 265)	1
  (6584, 154)	2
  (6584, 368)	2
  (6585, 387)	1
  (6585, 444)	1
  (6585, 409)	1
  (6585, 237)	1
  (6585, 354)	1
  (6585, 363)	1
  (6586, 444)	1
  (6586, 480)	1
  (6586, 278)	1
  (6586, 65)	1
  (6587, 212)	2
  (6587, 105)	1
  (6587, 284)	1
  (6587, 233)	1
  (6587, 308)	1
  (6587, 391)	1
  (6587, 196)	1
   someth  son  song
0       0    0     0
1       0    0     0
2       0    0     0


In [92]:
fourth_tweet = count_vectorized_df.iloc[3]
fourth_nonzero = fourth_tweet[fourth_tweet > 0]

print("Token counts:\n", fourth_nonzero)

Token counts:
 cant    1
deal    1
end     1
find    1
keep    1
like    1
may     1
say     1
talk    1
Name: 3, dtype: Sparse[int64, 0]


In [93]:
word_freq = count_vectorized_df.sum().sort_values(ascending=False)
print(word_freq.head(16))

tomorrow    1126
go           733
day          667
night        641
may          533
tonight      501
see          439
time         429
im           422
get          398
today        389
game         382
saturday     379
friday       375
sunday       368
come         357
dtype: Sparse[int64, 0]


In [94]:
count_vectorized_df['label'] = labels

print("\nFinal DataFrame (first 5 rows, last 3 cols + label)")
print(count_vectorized_df.iloc[:4, -2:])


Final DataFrame (first 5 rows, last 3 cols + label)
   your  label
0     0      1
1     0     -1
2     0     -1
3     0     -1
